# EXP-002 - H2: Categorical Features Restored

**Hypothesis H2** (pre-registered): restoring the categorical features the production pipeline
drops (label + frequency encoding + email split) yields private LB delta-AUC >= +0.020 over
EXP-001 (private >= 0.9077) - a larger gain than the H1 model swap delivered.

**Feature diff vs EXP-001** (the ONLY change; model and params identical):
- label encoding of every object-dtype column (missing = own category, unseen -> -1)
- frequency encoding of those columns + numeric-coded high-cardinality categoricals
  (card1, card2, card3, card5, addr1, addr2)
- email provider/suffix split of P_/R_emaildomain, then encoded like the rest

Canonical implementations live in `src/features/engineering.py` (unit-tested); the functions
below are inline mirrors (the kernel cannot import the repo package).

**Discipline**: encoders are fit on each fold's training months (Scheme B) and on the train
partition only (Scheme A model + submission) - never on scored rows.

**Outputs**: `holdout_pred_exp002.csv` (for the local DeLong test vs EXP-001) and
`submission.csv`. DeLong is computed off-notebook with `src/models/delong.py`.

**Anchors**: EXP-001 holdout 0.9124; LB 0.9134 / 0.8877 (SUB-002). Deterministic (seed 42):
re-running reproduces the recorded LB 0.9251 / 0.8968 (SUB-003) - no resubmission needed.

In [ ]:
import os
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

SPLIT_QUANTILE = 0.8
SECONDS_PER_MONTH = 86400 * 30.44
EXCLUDE_COLS = {"isFraud", "TransactionID", "TransactionDT"}
MISSING_TOKEN = "__missing__"
FREQ_NUMERIC_CATS = ["card1", "card2", "card3", "card5", "addr1", "addr2"]

LGB_PARAMS = dict(
    objective="binary",
    learning_rate=0.05,
    num_leaves=192,
    min_data_in_leaf=100,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    seed=42,
    n_jobs=-1,
    verbosity=-1,
)
MAX_ROUNDS = 5000
ES_PATIENCE = 200

ON_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
if ON_KAGGLE:
    hits = sorted(Path("/kaggle/input").rglob("train_transaction.csv"))
    if not hits:
        raise FileNotFoundError("Competition data not attached (Add Input -> Competitions).")
    DATA_DIR = hits[0].parent
else:
    DATA_DIR = Path("../../../data/raw")
print(f"Data dir: {DATA_DIR}")

## 0. Encoding functions - inline mirror of `src/features/engineering.py`

In [ ]:
def _as_str(values):
    return values.astype("object").where(values.notna(), MISSING_TOKEN).astype(str)

def frequency_encode(train_values, values):
    freq = _as_str(train_values).value_counts(normalize=True)
    return _as_str(values).map(freq).fillna(0.0).astype("float32")

def label_encode(train_values, values):
    cats = {v: i for i, v in enumerate(sorted(_as_str(train_values).unique()))}
    return _as_str(values).map(cats).fillna(-1).astype("int32")

def split_email_domain(values, prefix):
    parts = _as_str(values).str.split(".")
    return pd.DataFrame(
        {f"{prefix}_provider": parts.str[0], f"{prefix}_suffix": parts.str[-1]},
        index=values.index,
    )

def build_categorical_block(train_df, df, label_cols, freq_cols):
    out = pd.DataFrame(index=df.index)
    for col in label_cols:
        out[f"{col}_le"] = label_encode(train_df[col], df[col])
    for col in freq_cols:
        out[f"{col}_freq"] = frequency_encode(train_df[col], df[col])
    return out

## 1. Data, split, numeric base and categorical column lists

In [ ]:
train_transaction = pd.read_csv(DATA_DIR / "train_transaction.csv")
train_identity = pd.read_csv(DATA_DIR / "train_identity.csv")
df = train_transaction.merge(train_identity, on="TransactionID", how="left")
del train_transaction, train_identity

df["DT_M"] = (df["TransactionDT"] / SECONDS_PER_MONTH).astype(int)
cutoff = df["TransactionDT"].quantile(SPLIT_QUANTILE)
train_mask = (df["TransactionDT"] < cutoff).to_numpy()

for col, prefix in [("P_emaildomain", "P_email"), ("R_emaildomain", "R_email")]:
    df = pd.concat([df, split_email_domain(df[col], prefix)], axis=1)

numeric_features = [
    c for c in df.columns
    if df[c].dtype != "O" and c not in EXCLUDE_COLS and c != "DT_M"
]
label_cols = [c for c in df.columns if df[c].dtype == "O"]
freq_cols = label_cols + FREQ_NUMERIC_CATS
print(f"Numeric base: {len(numeric_features)} | label-encoded: {len(label_cols)} | freq-encoded: {len(freq_cols)}")

X_num = df[numeric_features].astype("float32")
y = df["isFraud"].astype(int).to_numpy()
months = df["DT_M"].to_numpy()
trans_ids = df["TransactionID"].to_numpy()
cat_source = df[label_cols + FREQ_NUMERIC_CATS].copy()
del df

def make_X(fit_mask):
    block = build_categorical_block(cat_source[fit_mask], cat_source, label_cols, freq_cols)
    return pd.concat([X_num.reset_index(drop=True), block.reset_index(drop=True)], axis=1)

## 2. Scheme B - month-wise GroupKFold (7 folds), encoders fit per fold

In [ ]:
fold_aucs, fold_best_iters = [], []
t0 = time.time()
for m in sorted(set(months)):
    tr = months != m
    va = ~tr
    X_fold = make_X(fit_mask=tr)  # encoders fit on this fold's training months
    clf = lgb.LGBMClassifier(n_estimators=MAX_ROUNDS, **LGB_PARAMS)
    clf.fit(
        X_fold[tr], y[tr],
        eval_set=[(X_fold[va], y[va])],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(ES_PATIENCE, verbose=False), lgb.log_evaluation(0)],
    )
    auc = roc_auc_score(y[va], clf.predict_proba(X_fold[va])[:, 1])
    fold_aucs.append(auc)
    fold_best_iters.append(clf.best_iteration_)
    del X_fold
    print(f"fold month={m}: AUC={auc:.4f}  best_iter={clf.best_iteration_}  ({(time.time()-t0)/60:.1f} min elapsed)")

scheme_b_mean, scheme_b_std = float(np.mean(fold_aucs)), float(np.std(fold_aucs))
print(f"\nScheme B GroupKFold: {scheme_b_mean:.4f} +/- {scheme_b_std:.4f}")

## 3. Scheme A model - ES inside the train partition, refit at best_iter * 1.1

In [ ]:
X_final = make_X(fit_mask=train_mask)  # encoders fit on train partition only

train_months = months[train_mask]
es_month = train_months.max()
sub_tr = train_mask & (months < es_month)
es_va = train_mask & (months == es_month)

es_clf = lgb.LGBMClassifier(n_estimators=MAX_ROUNDS, **LGB_PARAMS)
es_clf.fit(
    X_final[sub_tr], y[sub_tr],
    eval_set=[(X_final[es_va], y[es_va])],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(ES_PATIENCE, verbose=False), lgb.log_evaluation(0)],
)
best_iter = es_clf.best_iteration_
final_rounds = max(int(best_iter * 1.1), 100)
print(f"ES month: {es_month} | best_iter: {best_iter} | refit rounds: {final_rounds}")

model = lgb.LGBMClassifier(n_estimators=final_rounds, **LGB_PARAMS)
model.fit(X_final[train_mask], y[train_mask])

val_proba = model.predict_proba(X_final[~train_mask])[:, 1]
holdout_auc = roc_auc_score(y[~train_mask], val_proba)
print(f"Scheme A holdout ROC-AUC: {holdout_auc:.4f}  (EXP-001: 0.9124)")

## 4. Save holdout predictions and submission

In [ ]:
pd.DataFrame(
    {"TransactionID": trans_ids[~train_mask], "y_true": y[~train_mask], "score": val_proba}
).to_csv("holdout_pred_exp002.csv", index=False)
print("Saved holdout_pred_exp002.csv")

test_transaction = pd.read_csv(DATA_DIR / "test_transaction.csv")
test_identity = pd.read_csv(DATA_DIR / "test_identity.csv")
test_identity.columns = [c.replace("id-", "id_") for c in test_identity.columns]
df_test = test_transaction.merge(test_identity, on="TransactionID", how="left")
del test_transaction, test_identity

for col, prefix in [("P_emaildomain", "P_email"), ("R_emaildomain", "R_email")]:
    df_test = pd.concat([df_test, split_email_domain(df_test[col], prefix)], axis=1)

X_test_num = df_test.reindex(columns=numeric_features).astype("float32")
test_block = build_categorical_block(
    cat_source[train_mask], df_test.reindex(columns=label_cols + FREQ_NUMERIC_CATS),
    label_cols, freq_cols,
)
X_test = pd.concat([X_test_num.reset_index(drop=True), test_block.reset_index(drop=True)], axis=1)
assert list(X_test.columns) == list(X_final.columns), "feature contract violated"

test_proba = model.predict_proba(X_test)[:, 1]
pd.DataFrame({"TransactionID": df_test["TransactionID"], "isFraud": test_proba}).to_csv(
    "submission.csv", index=False
)
print(f"Saved submission.csv ({len(df_test):,} rows, expected 506,691)")

print("\n=== EXP-002 summary ===")
print(f"Scheme A holdout AUC : {holdout_auc:.4f}")
print(f"Scheme B GroupKFold  : {scheme_b_mean:.4f} +/- {scheme_b_std:.4f}")
print(f"Per-fold AUCs        : {[round(float(a), 4) for a in fold_aucs]}")